# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shehzadi434/flyrank-Internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one content item (page) on one day (`content_hash_id` + `client_hash_id` + `report_date`).

**Development window:** March 2026 (`month=2026-03`) — a mid-panel month, not the final month (June 2026).

**Final model window:** Feature window = 90 days (March–May 2026); Label window = 30 days (June 2026). This follows the past→future pattern from Notebook 02.

**Output:** A ranked list of pages to review for refresh, measured by Precision@50.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. Ready to Go.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. Ready to Go.


In [15]:
# Verify grain and time window
import duckdb
from google.colab import userdata

# Connect to DuckDB and authenticate
con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Query 1: Verify grain — one row = one content item per day
print("=" * 50)
print("QUERY 1: Verify grain")
print("=" * 50)

grain_check = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

if len(grain_check) == 0:
    print(" Grain confirmed: One row = one content item per day")
else:
    print(" Grain violation found!")

# Query 2: Date range and row count
print("\n" + "=" * 50)
print("QUERY 2: Date range and row count")
print("=" * 50)

result = con.sql(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(f"March 2026 slice:")
print(f"  Date range: {result['min_date'].iloc[0]} to {result['max_date'].iloc[0]}")
print(f"  Row count: {result['row_count'].iloc[0]:,}")
print(f"  Unique pages: {result['unique_pages'].iloc[0]:,}")
print(f"  Unique clients: {result['unique_clients'].iloc[0]:,}")

QUERY 1: Verify grain


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Grain confirmed: One row = one content item per day

QUERY 2: Date range and row count
March 2026 slice:
  Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
  Row count: 9,841,378
  Unique pages: 331,437
  Unique clients: 55


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


| Bucket | Fields | Why |
|--------|--------|-----|
| **Features** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `content_age_days` | Observable signals known BEFORE the decision point. |
| **Label / Proxy** | `is_declining` — impressions dropped >20% month-over-month | Target to predict based on future outcome. |
| **Context** | `content_hash_id`, `client_hash_id`, `report_date` | Grouping, joins, and client-holdout validation only. |
| **Excluded** | `trend_pct`, `trend_direction` (leakage); product decision flags | These encode the label in disguise (Notebook 02 lesson). Product flags are excluded by design. |

In [16]:
# Show available columns and their types
print("=" * 50)
print("Available columns in fact_content_daily_performance (March 2026)")
print("=" * 50)

columns = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").df()

print(columns[['column_name', 'column_type']].to_string(index=False))

print("\n" + "=" * 50)
print("Key fields for my lane (Lane 1 — Content Refresh Prediction)")
print("=" * 50)
print("""
Features (knowable before decision):
  - gsc_impressions: search impressions count
  - gsc_clicks: search clicks count
  - gsc_avg_position: average search position
  - content_age_days: days since content creation

Label:
  - is_declining: impressions dropped >20% month-over-month

Context (grouping only):
  - content_hash_id, client_hash_id, report_date

Excluded (leakage):
  - Any column that encodes the label in disguise
  - Product decision flags (not in our data)
""")

Available columns in fact_content_daily_performance (March 2026)
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
          

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Three verification queries:**

1. **Availability with IS TRUE:** Filter GSC/GA4 flags properly. The flyrank-data skill warns: rows before a client's GA4 start have zero-filled columns with `ga4_data_available = FALSE`. Always use `IS TRUE` / `IS NOT TRUE`.

2. **Missing values check:** Position is missing for 63.31% of rows — handle with care.

3. **Per-client history:** Unbalanced panel confirmed — 52 clients have full month, some have less. Use client-holdout validation.

In [17]:
print("=" * 50)
print("QUERY 1: Availability with IS TRUE")
print("=" * 50)

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS has_impressions
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(availability)
print(f"\n GSC available (IS TRUE): {availability['gsc_available'].iloc[0]:,} rows")
print(f"   GA4 available (IS TRUE): {availability['ga4_available'].iloc[0]:,} rows")


print("\n" + "=" * 50)
print("QUERY 2: Missing values check by column")
print("=" * 50)

missing_check = con.sql(f"""
    SELECT
        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS impressions_missing,
        AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS clicks_missing,
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS position_missing
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(f"Impressions missing: {missing_check['impressions_missing'].iloc[0]:.2%}")
print(f"Clicks missing: {missing_check['clicks_missing'].iloc[0]:.2%}")
print(f"Position missing: {missing_check['position_missing'].iloc[0]:.2%}")


print("\n" + "=" * 50)
print("QUERY 3: Per-client history depth (unbalanced panel check)")
print("=" * 50)

history_check = con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(DISTINCT report_date) AS days_in_data,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id
    ORDER BY days_in_data DESC
    LIMIT 10
""").df()

print(history_check)
print("\n Client history depth varies — this is an unbalanced panel.")
print("   Models should use client-holdout validation to test generalization.")

QUERY 1: Availability with IS TRUE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available  ga4_available  has_impressions
0     9841378      3611061.0       413966.0        3611061.0

 GSC available (IS TRUE): 3,611,061.0 rows
   GA4 available (IS TRUE): 413,966.0 rows

QUERY 2: Missing values check by column


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Impressions missing: 0.00%
Clicks missing: 0.00%
Position missing: 63.31%

QUERY 3: Per-client history depth (unbalanced panel check)
            client_hash_id  days_in_data first_date  last_date
0  client_73cda7b4e4f265ea            31 2026-03-01 2026-03-31
1  client_c182d11e4862a37d            31 2026-03-01 2026-03-31
2  client_f623b01661d4bfe4            31 2026-03-01 2026-03-31
3  client_8ae2bfb5aa1ffa1e            31 2026-03-01 2026-03-31
4  client_a2eeb8899886adde            31 2026-03-01 2026-03-31
5  client_d211cb07b9059bab            31 2026-03-01 2026-03-31
6  client_4a18d1793d92fb84            31 2026-03-01 2026-03-31
7  client_62f4a7e64f5e0096            31 2026-03-01 2026-03-31
8  client_fef1a8f436438636            31 2026-03-01 2026-03-31
9  client_ba65e80a1116ae41            31 2026-03-01 2026-03-31

 Client history depth varies — this is an unbalanced panel.
   Models should use client-holdout validation to test generalization.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can NEVER tell me:**

1. **Causality:** Observed patterns ≠ proof that refreshing causes recovery. That requires an experiment.

2. **Google's algorithm:** Data shows what happened, not why.

3. **Why a page is declining:** Seasonality, consolidation, or real decline — data alone can't always distinguish.

4. **Future performance on new clients:** Model is built on this dataset; performance on unseen clients is unknown.

5. **Client-specific insights for short-history clients:** Unbalanced panel means some clients have limited data.

**Specific limitation:** March 2026 is one month — model may not generalize to other months without retesting. The `_sample` table (June 2026) is the final month — treat it as a sealed test month, never development data.

In [19]:
print("=" * 50)
print("DATA LIMIT: Unbalanced panel demonstration")
print("=" * 50)

# Show the distribution of client history lengths
history_dist = con.sql(f"""
    WITH client_history AS (
        SELECT
            client_hash_id,
            COUNT(DISTINCT report_date) AS days_in_data
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY client_hash_id
    )
    SELECT
        CASE
            WHEN days_in_data <= 10 THEN '1-10 days'
            WHEN days_in_data <= 20 THEN '11-20 days'
            WHEN days_in_data <= 28 THEN '21-28 days'
            ELSE '29-31 days (full month)'
        END AS history_category,
        COUNT(*) AS client_count
    FROM client_history
    GROUP BY history_category
    ORDER BY history_category
""").df()

print("Client history depth distribution:")
print(history_dist)

print("\n Not all clients have the same amount of history.")
print(" Models must use client-holdout validation to avoid overfitting to specific clients.")

DATA LIMIT: Unbalanced panel demonstration
Client history depth distribution:
          history_category  client_count
0                1-10 days             1
1               11-20 days             2
2  29-31 days (full month)            52

 Not all clients have the same amount of history.
 Models must use client-holdout validation to avoid overfitting to specific clients.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.